# 프로젝트: mini BERT 만들기

vocab size = **8000**, 전체 파라미터 **~1M**, **10 Epoch** 학습

---
## 목차
1. 라이브러리 설치 및 임포트
2. Tokenizer 준비 (SentencePiece, vocab_size=8000)
3. 데이터 전처리 (1) - MASK 생성
4. 데이터 전처리 (2) - NSP pair 생성
5. 데이터 전처리 (3) - 데이터셋 완성 (np.memmap + JSON)
6. BERT 모델 구현 (mini: ~1M params)
7. pretrain 진행 (10 Epoch, Cosine LR Schedule)
8. 프로젝트 결과 시각화

---
## Step 0. 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install sentencepiece tqdm torchinfo
!conda install -y -c conda-forge ipywidgets 2>/dev/null || true

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import os
import re
import math
import numpy as np
import json
import copy
import random
import collections
import shutil

import matplotlib.pyplot as plt
import sentencepiece as spm
from tqdm.notebook import tqdm

# 재현성 설정
random_seed = 1234
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")

In [ ]:
# 데이터 디렉토리 생성 및 kowiki 다운로드
!mkdir -p ~/work/bert_pretrain/data
!mkdir -p ~/work/bert_pretrain/models

import os
DATA_DIR   = os.path.expanduser('~/work/bert_pretrain/data')
MODEL_DIR  = os.path.expanduser('~/work/bert_pretrain/models')
corpus_file = os.path.join(DATA_DIR, 'kowiki.txt')

if not os.path.exists(corpus_file):
    print("kowiki.txt 다운로드 중...")
    !wget -q https://aiffelstaticprd.blob.core.windows.net/media/documents/kowiki.txt.zip -P {DATA_DIR}
    !cd {DATA_DIR} && unzip -q kowiki.txt.zip
    print("완료!")
else:
    print("kowiki.txt 이미 존재합니다.")

print(f"코퍼스 경로: {corpus_file}")

---
## Step 1. Tokenizer 준비
SentencePiece BPE 모델 학습 (vocab_size=8000)

- BERT 필수 특수 토큰 포함: `[PAD]`, `[UNK]`, `[BOS]`, `[EOS]`, `[SEP]`, `[CLS]`, `[MASK]`
- `[PAD]=0, [UNK]=1, [BOS]=2, [EOS]=3` 로 고정

In [ ]:
import sentencepiece as spm

# ─── 하이퍼파라미터 ───────────────────────────────────────────
VOCAB_SIZE  = 8000    # ★ mini BERT 요구사항
N_SEQ       = 128     # 시퀀스 최대 길이 (짧게 → 파라미터↓)
# ─────────────────────────────────────────────────────────────

prefix_8k = os.path.join(DATA_DIR, 'ko_8000')
spm_model_file = prefix_8k + '.model'

if not os.path.exists(spm_model_file):
    print("SentencePiece 모델 학습 중... (수 분 소요)")
    spm.SentencePieceTrainer.train(
        f"--input={corpus_file}"
        f" --model_prefix={prefix_8k}"
        f" --vocab_size={VOCAB_SIZE + 7}"
        f" --model_type=bpe"
        f" --max_sentence_length=999999"
        f" --pad_id=0 --pad_piece=[PAD]"
        f" --unk_id=1 --unk_piece=[UNK]"
        f" --bos_id=2 --bos_piece=[BOS]"
        f" --eos_id=3 --eos_piece=[EOS]"
        f" --user_defined_symbols=[SEP],[CLS],[MASK]"
    )
    print("SentencePiece 학습 완료!")
else:
    print("SentencePiece 모델 이미 존재합니다.")

# vocab 로드
vocab = spm.SentencePieceProcessor()
vocab.load(spm_model_file)
print(f"Vocab size : {vocab.get_piece_size()}")

# 특수 토큰 ID 확인
i_pad  = vocab.piece_to_id('[PAD]')
i_cls  = vocab.piece_to_id('[CLS]')
i_sep  = vocab.piece_to_id('[SEP]')
i_mask = vocab.piece_to_id('[MASK]')
print(f"[PAD]={i_pad}, [CLS]={i_cls}, [SEP]={i_sep}, [MASK]={i_mask}")

# vocab list (random token 치환용, 특수 토큰 제외)
vocab_list = [vocab.id_to_piece(i) for i in range(vocab.get_piece_size())
              if vocab.id_to_piece(i) not in ['[PAD]','[UNK]','[BOS]','[EOS]','[SEP]','[CLS]','[MASK]']]
print(f"vocab_list (특수토큰 제외) 크기: {len(vocab_list)}")

In [ ]:
# 간단 동작 테스트
test_text = "한국어 위키피디아로 mini BERT를 학습합니다."
pieces = vocab.encode_as_pieces(test_text)
ids    = vocab.encode_as_ids(test_text)
print("조각:", pieces)
print("ID  :", ids)

---
## Step 2. 데이터 전처리 (1) - MASK 생성

MLM 규칙:
- 전체 토큰의 **15%** 를 마스킹 대상으로 선정
- 80% → `[MASK]` 치환
- 10% → 랜덤 토큰 치환
- 10% → 원래 토큰 유지
- **단어(띄어쓰기) 단위로** 마스킹

In [ ]:
def create_pretrain_mask(tokens, mask_cnt, vocab_list):
    """
    MLM 마스크 생성
    :param tokens    : token 리스트 ([CLS] sent_a [SEP] sent_b [SEP])
    :param mask_cnt  : 마스킹할 토큰 수 (전체의 15%)
    :param vocab_list: 랜덤 치환용 vocab 리스트
    :return tokens   : 마스크된 tokens
    :return mask_idx : 마스크된 인덱스 리스트
    :return mask_label: 원래 token 리스트
    """
    # ① 단어(띄어쓰기) 단위로 후보 인덱스 묶기
    cand_idx = []
    for i, token in enumerate(tokens):
        if token in ("[CLS]", "[SEP]"):
            continue
        # '▁' (U+2581) 로 시작하면 새 단어
        if 0 < len(cand_idx) and not token.startswith(u"\u2581"):
            cand_idx[-1].append(i)
        else:
            cand_idx.append([i])

    # ② 순서 셔플
    random.shuffle(cand_idx)

    mask_lms = []
    for index_set in cand_idx:
        if len(mask_lms) >= mask_cnt:
            break
        if len(mask_lms) + len(index_set) > mask_cnt:
            continue
        dice = random.random()
        for index in index_set:
            if dice < 0.8:
                masked_token = "[MASK]"
            elif dice < 0.9:
                masked_token = tokens[index]       # 원래 토큰 유지
            else:
                masked_token = random.choice(vocab_list)  # 랜덤 치환
            mask_lms.append({"index": index, "label": tokens[index]})
            tokens[index] = masked_token

    # ③ 인덱스 정렬
    mask_lms = sorted(mask_lms, key=lambda x: x["index"])
    mask_idx   = [p["index"] for p in mask_lms]
    mask_label = [p["label"] for p in mask_lms]
    return tokens, mask_idx, mask_label

print("슝=3")

In [ ]:
# 동작 테스트
sample_text = "추적추적 비가 내리는 날이었어 그날은 왠지 손님이 많아 첫 번에 삼십 전 둘째 번 오십 전 전짜리 백통화 서푼에 손바닥 위엔 기쁨의 눈물이 흘러"
sample_tokens = ['[CLS]'] + vocab.encode_as_pieces(sample_text[:60]) + ['[SEP]'] \
              + vocab.encode_as_pieces(sample_text[60:]) + ['[SEP]']
tokens_org = copy.deepcopy(sample_tokens)
mask_cnt = int((len(tokens_org) - 3) * 0.15)

tokens_masked, mask_idx, mask_label = create_pretrain_mask(
    copy.deepcopy(tokens_org), mask_cnt, vocab_list
)
print(f"원본 tokens  ({len(tokens_org)}):", tokens_org[:15], "...")
print(f"마스크 tokens ({len(tokens_masked)}):", tokens_masked[:15], "...")
print(f"mask_idx  : {mask_idx}")
print(f"mask_label: {mask_label}")

---
## Step 3. 데이터 전처리 (2) - NSP pair 생성

- 50% 확률로 **연속(is_next=1)** / **비연속(is_next=0)** 쌍 생성
- `[CLS] + sent_A + [SEP] + sent_B + [SEP]` 구성
- segment: sent_A → 0, sent_B → 1
- MLM + NSP 동시에 처리

In [ ]:
def trim_tokens(tokens_a, tokens_b, max_seq):
    """tokens_a + tokens_b의 길이가 max_seq를 넘지 않도록 번갈아가며 자름"""
    while True:
        total = len(tokens_a) + len(tokens_b)
        if total <= max_seq:
            break
        if len(tokens_a) > len(tokens_b):
            tokens_a.pop()
        else:
            tokens_b.pop()


def create_pretrain_instances(vocab, doc, n_seq, mask_prob, vocab_list):
    """
    문서(doc) 한 개에서 pretrain 인스턴스(NSP+MLM) 생성
    :param vocab      : SentencePiece 모델
    :param doc        : [[token, ...], [token, ...], ...]  (문장 토큰 리스트의 리스트)
    :param n_seq      : 최대 시퀀스 길이 (N_SEQ)
    :param mask_prob  : 마스킹 비율 (0.15)
    :param vocab_list : 랜덤 치환용 vocab 리스트
    :return instances : 인스턴스 딕셔너리 리스트
    """
    max_seq = n_seq - 3  # [CLS], [SEP], [SEP] 3개 제외
    instances = []
    current_chunk  = []
    current_length = 0

    for i in range(len(doc)):
        current_chunk.append(doc[i])
        current_length += len(doc[i])

        if 1 < len(current_chunk) and (i == len(doc) - 1 or current_length >= max_seq):
            # tokens_a 분할
            a_end = random.randrange(1, len(current_chunk)) if 1 < len(current_chunk) else 1
            tokens_a = []
            for j in range(a_end):
                tokens_a.extend(current_chunk[j])
            tokens_b = []
            for j in range(a_end, len(current_chunk)):
                tokens_b.extend(current_chunk[j])

            # 50% swap → NSP False
            if random.random() < 0.5:
                is_next = 0
                tokens_a, tokens_b = tokens_b, tokens_a
            else:
                is_next = 1

            trim_tokens(tokens_a, tokens_b, max_seq)
            assert 0 < len(tokens_a)
            assert 0 < len(tokens_b)

            # [CLS] + A + [SEP] + B + [SEP]
            tokens  = ["[CLS]"] + tokens_a + ["[SEP]"] + tokens_b + ["[SEP]"]
            segment = [0] * (len(tokens_a) + 2) + [1] * (len(tokens_b) + 1)

            # MLM 마스킹
            tokens, mask_idx, mask_label = create_pretrain_mask(
                tokens, int((len(tokens) - 3) * mask_prob), vocab_list
            )

            instances.append({
                "tokens"    : tokens,
                "segment"   : segment,
                "is_next"   : is_next,
                "mask_idx"  : mask_idx,
                "mask_label": mask_label
            })

            current_chunk  = []
            current_length = 0

    return instances

print("슝=3")

In [ ]:
# 동작 테스트 (소규모 doc)
test_doc = [vocab.encode_as_pieces(s) for s in [
    "추적추적 비가 내리는 날이었어.",
    "그날은 왠지 손님이 많아 첫 번에 삼십 전.",
    "둘째 번 오십 전 전짜리 백통화 서푼에.",
    "손바닥 위엔 기쁨의 눈물이 흘러.",
    "컬컬한 목에 모주 한 잔을 적셔."
]]

test_instances = create_pretrain_instances(vocab, test_doc, N_SEQ, 0.15, vocab_list)
print(f"생성된 인스턴스 수: {len(test_instances)}")
for inst in test_instances:
    print(f"  is_next={inst['is_next']}, len={len(inst['tokens'])}, mask={inst['mask_idx']}")

---
## Step 4. 데이터 전처리 (3) - 데이터셋 완성 (np.memmap + JSON)

In [ ]:
def save_pretrain_data(vocab, corpus_file, json_file, n_seq, mask_prob, vocab_list,
                       max_docs=None):
    """
    전체 코퍼스를 읽어 pretrain JSON 저장
    :param max_docs: 테스트용 제한 (None이면 전체)
    """
    total_instances = []

    # 코퍼스 총 줄 수 계산
    with open(corpus_file, 'r') as f:
        total_lines = sum(1 for _ in f)

    doc_count = 0
    with open(corpus_file, 'r') as in_f:
        doc = []
        for line in tqdm(in_f, total=total_lines, desc="읽는 중"):
            line = line.strip()
            if line == "":  # 빈 줄 = 새 문서
                if 0 < len(doc):
                    instances = create_pretrain_instances(
                        vocab, doc, n_seq, mask_prob, vocab_list
                    )
                    total_instances.extend(instances)
                    doc = []
                    doc_count += 1
                    if max_docs and doc_count >= max_docs:
                        break
            else:
                pieces = vocab.encode_as_pieces(line)
                if pieces:
                    doc.append(pieces)
        if doc:  # 마지막 문서 처리
            total_instances.extend(
                create_pretrain_instances(vocab, doc, n_seq, mask_prob, vocab_list)
            )

    # JSON 저장
    with open(json_file, 'w', encoding='utf-8') as out_f:
        json.dump(total_instances, out_f, ensure_ascii=False)

    print(f"저장 완료 | 총 문서: {doc_count} | 총 인스턴스: {len(total_instances)}")
    print(f"파일: {json_file}")
    return total_instances

print("슝=3")

In [ ]:
# np.memmap으로 저장된 데이터 로드용 함수
def instance_to_array(instance, n_seq, n_vocab, vocab):
    """
    인스턴스 딕셔너리를 numpy 배열로 변환 (패딩 포함)
    반환: enc_tokens, segments, labels_nsp, labels_mlm
    """
    tokens    = instance['tokens']
    segment   = instance['segment']
    is_next   = instance['is_next']
    mask_idx  = instance['mask_idx']
    mask_label= instance['mask_label']

    # token → id
    token_ids = vocab.piece_to_id(tokens)  # list of ids
    seg_ids   = segment

    # MLM label array (마스크된 위치만 실제 ID, 나머지는 0)
    mlm_labels = [0] * len(tokens)
    for idx, label in zip(mask_idx, mask_label):
        mlm_labels[idx] = vocab.piece_to_id(label)

    # 패딩
    pad_len = n_seq - len(token_ids)
    enc_tokens_arr  = token_ids  + [0] * pad_len
    segments_arr    = seg_ids    + [0] * pad_len
    labels_mlm_arr  = mlm_labels + [0] * pad_len

    return (
        np.array(enc_tokens_arr,  dtype=np.int32),
        np.array(segments_arr,    dtype=np.int32),
        np.int32(is_next),
        np.array(labels_mlm_arr,  dtype=np.int32)
    )

print("슝=3")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 데이터셋 생성 (시간 절약을 위해 max_docs=5000으로 제한 가능)
# 전체 학습이 필요하면 max_docs=None
# ─────────────────────────────────────────────────────────────
MAX_DOCS  = 5000    # ← 테스트: 5000, 전체: None
json_file = os.path.join(DATA_DIR, 'pretrain_8000.json')

if not os.path.exists(json_file):
    instances = save_pretrain_data(
        vocab, corpus_file, json_file,
        n_seq=N_SEQ, mask_prob=0.15,
        vocab_list=vocab_list, max_docs=MAX_DOCS
    )
else:
    print(f"JSON 파일 이미 존재: {json_file}")
    with open(json_file, 'r') as f:
        instances = json.load(f)
    print(f"로드된 인스턴스 수: {len(instances)}")

In [ ]:
# np.memmap 으로 배열 저장 (메모리 효율화)
memmap_file = os.path.join(DATA_DIR, 'pretrain_8000.dat')
N_INST = len(instances)

print(f"총 인스턴스: {N_INST}, N_SEQ: {N_SEQ}")

if not os.path.exists(memmap_file):
    # shape: (N_INST, 4, N_SEQ) — enc_tokens / segments / labels_mlm + nsp
    # 컬럼 0: enc_tokens, 1: segments, 2: labels_nsp(repeat), 3: labels_mlm
    mm = np.memmap(memmap_file, dtype=np.int32, mode='w+', shape=(N_INST, 4, N_SEQ))

    for i, inst in enumerate(tqdm(instances, desc='memmap 저장')):
        enc, seg, nsp, mlm = instance_to_array(inst, N_SEQ, VOCAB_SIZE, vocab)
        mm[i, 0] = enc
        mm[i, 1] = seg
        mm[i, 2] = np.full(N_SEQ, nsp, dtype=np.int32)
        mm[i, 3] = mlm

    mm.flush()
    print(f"memmap 저장 완료: {memmap_file}")
else:
    print(f"memmap 파일 이미 존재: {memmap_file}")

# 로드 확인
mm = np.memmap(memmap_file, dtype=np.int32, mode='r', shape=(N_INST, 4, N_SEQ))
print(f"memmap shape: {mm.shape}")
print("샘플 enc_tokens:", mm[0, 0, :20])
print("샘플 is_next   :", mm[0, 2, 0])

---
## Step 5. BERT 모델 구현 (mini ~1M params)

### ★ mini BERT 구성
| 설정 | 값 |
|---|---|
| vocab_size | 8,000 |
| d_model | 128 |
| n_head | 2 |
| d_head | 64 |
| d_ff | 512 |
| n_layer | 2 |
| n_seq | 128 |
| 총 파라미터 | ~1M |

In [ ]:
# ── 헬퍼 함수 ───────────────────────────────────────────────

def get_pad_mask(tokens, i_pad=0):
    """
    pad mask: pad 위치 = 1, 나머지 = 0
    :param tokens: (bs, n_seq)
    :return mask : (bs, 1, n_seq)
    """
    mask = (tokens == i_pad).float()
    mask = mask.unsqueeze(1)
    return mask


def get_ahead_mask(tokens, i_pad=0):
    """
    ahead(causal) + pad mask
    :param tokens: (bs, n_seq)
    :return mask : (bs, n_seq, n_seq)
    """
    n_seq = tokens.size(1)
    ahead_mask = 1 - torch.tril(torch.ones((n_seq, n_seq), device=tokens.device))
    ahead_mask = ahead_mask.unsqueeze(0)
    pad_mask   = get_pad_mask(tokens, i_pad)
    mask = torch.maximum(ahead_mask, pad_mask)
    return mask


def gelu(x):
    """GELU activation"""
    return 0.5 * x * (1 + torch.tanh(
        math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
    ))


class Config(dict):
    """JSON → Config 객체"""
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__

    @classmethod
    def load(cls, file):
        with open(file, 'r') as f:
            return cls(json.loads(f.read()))

print("슝=3")

In [ ]:
# ── mini BERT config ──────────────────────────────────────────
config = Config({
    "n_vocab"          : VOCAB_SIZE + 7,  # spm 학습 시 +7 했으므로
    "d_model"          : 128,
    "n_head"           : 2,
    "d_head"           : 64,
    "d_ff"             : 512,
    "n_layer"          : 2,
    "n_seq"            : N_SEQ,
    "dropout"          : 0.1,
    "layernorm_epsilon": 1e-12,
    "i_pad"            : i_pad,
})
print("config:", dict(config))

In [ ]:
# ── Embedding 레이어 ─────────────────────────────────────────

class SharedEmbedding(nn.Module):
    """Weight-tied Embedding (Input Embedding = Output Projection)"""
    def __init__(self, config):
        super().__init__()
        self.n_vocab  = config.n_vocab
        self.d_model  = config.d_model
        self.weight   = nn.Parameter(torch.empty(self.n_vocab, self.d_model))
        nn.init.trunc_normal_(self.weight, std=0.02)

    def forward(self, inputs, mode="embedding"):
        if mode == "embedding":
            return self._embedding(inputs)
        elif mode == "linear":
            return self._linear(inputs)
        raise ValueError(f"mode {mode} is not valid.")

    def _embedding(self, inputs):
        inputs = torch.clamp(inputs, max=self.weight.size(0) - 1)
        return self.weight[inputs.long()]

    def _linear(self, inputs):   # (bs, n_seq, d_model) → (bs, n_seq, n_vocab)
        bs, n_seq, _ = inputs.shape
        out = torch.matmul(inputs.view(-1, self.d_model), self.weight.T)
        return out.view(bs, n_seq, self.n_vocab)


class PositionEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed = nn.Embedding(config.n_seq, config.d_model)
        nn.init.trunc_normal_(self.embed.weight, std=0.02)

    def forward(self, inputs):
        pos = torch.cumsum(torch.ones_like(inputs), dim=1) - 1
        return self.embed(pos.long())

print("슝=3")

In [ ]:
# ── Transformer Encoder 레이어 ───────────────────────────────

class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, attn_mask):
        scale       = math.sqrt(K.shape[-1])
        attn_score  = torch.matmul(Q, K.transpose(-2, -1)) / scale
        attn_score  = attn_score - attn_mask * 1e9
        attn_prob   = F.softmax(attn_score, dim=-1)
        return torch.matmul(attn_prob, V)


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.d_head = config.d_head
        self.W_Q = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_K = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_V = nn.Linear(config.d_model, config.n_head * config.d_head)
        self.W_O = nn.Linear(config.n_head * config.d_head, config.d_model)
        self.attn = ScaleDotProductAttention()

    def forward(self, Q, K, V, attn_mask):
        bs = Q.shape[0]
        Q_m = self.W_Q(Q).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
        K_m = self.W_K(K).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
        V_m = self.W_V(V).view(bs, -1, self.n_head, self.d_head).transpose(1, 2)
        out = self.attn(Q_m, K_m, V_m, attn_mask.unsqueeze(1))
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.n_head * self.d_head)
        return self.W_O(out)


class PositionWiseFeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.W1   = nn.Linear(config.d_model, config.d_ff)
        self.W2   = nn.Linear(config.d_ff,   config.d_model)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.W2(self.act(self.W1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn    = MultiHeadAttention(config)
        self.norm1   = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.ffn     = PositionWiseFeedForward(config)
        self.norm2   = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask):
        attn_out = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

print("슝=3")

In [ ]:
# ── BERT 레이어 ─────────────────────────────────────────────

class BERT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.i_pad    = config.i_pad
        self.tok_emb  = SharedEmbedding(config)
        self.pos_emb  = PositionEmbedding(config)
        self.seg_emb  = nn.Embedding(2, config.d_model)
        nn.init.trunc_normal_(self.seg_emb.weight, std=0.02)
        self.norm     = nn.LayerNorm(config.d_model, eps=config.layernorm_epsilon)
        self.drop     = nn.Dropout(config.dropout)
        self.layers   = nn.ModuleList([EncoderLayer(config) for _ in range(config.n_layer)])

    def forward(self, tokens, segments):
        mask = get_pad_mask(tokens, self.i_pad)         # (bs, 1, n_seq)
        emb  = self.tok_emb(tokens) + self.pos_emb(tokens) + self.seg_emb(segments.long())
        x    = self.drop(self.norm(emb))
        for layer in self.layers:
            x = layer(x, mask)
        logits_cls = x[:, 0]                            # [CLS] 토큰
        logits_lm  = self.tok_emb(x, mode="linear")    # Weight-tied projection
        return logits_cls, logits_lm


class PooledOutput(nn.Module):
    """NSP 헤드"""
    def __init__(self, config, n_output=2):
        super().__init__()
        self.dense1 = nn.Linear(config.d_model, config.d_model)
        self.dense2 = nn.Linear(config.d_model, n_output, bias=False)

    def forward(self, x):
        return self.dense2(torch.tanh(self.dense1(x)))


class MiniBERTPretrain(nn.Module):
    """mini BERT Pretrain 모델 (MLM + NSP)"""
    def __init__(self, config):
        super().__init__()
        self.bert = BERT(config)
        self.nsp  = PooledOutput(config, n_output=2)

    def forward(self, tokens, segments):
        tokens   = tokens.long()
        segments = segments.long()
        logits_cls, logits_lm = self.bert(tokens, segments)
        logits_nsp = self.nsp(logits_cls)   # (bs, 2)
        return logits_nsp, logits_lm        # logits_lm: (bs, n_seq, n_vocab)

print("슝=3")

In [ ]:
# ── 파라미터 수 확인 ─────────────────────────────────────────
from torchinfo import summary

model = MiniBERTPretrain(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 파라미터     : {total_params:,}")
print(f"학습 가능 파라미터: {trainable_params:,}")
print(f"목표: ~1,000,000  {'✅' if 700_000 <= total_params <= 1_500_000 else '⚠️ 조정 필요'}")

# torchinfo 상세 요약
dummy_tokens   = torch.randint(0, config.n_vocab, (2, N_SEQ)).to(device)
dummy_segments = torch.zeros(2, N_SEQ, dtype=torch.long).to(device)
summary(model, input_data=(dummy_tokens, dummy_segments), device=device)

---
## Step 6. pretrain 진행
- Loss: MLM CrossEntropy (pad 제외, ×20 가중치) + NSP CrossEntropy
- LR: Cosine Schedule (warmup 포함)
- **10 Epoch**

In [ ]:
# ── Loss / Accuracy 함수 ────────────────────────────────────

def mlm_loss(y_true, y_pred):
    """
    MLM loss (pad=0 위치 제외)
    :param y_true: (bs, n_seq)          정답 ID (마스크 위치만 실제 ID, 나머지 0)
    :param y_pred: (bs, n_seq, n_vocab) logits
    """
    loss = F.cross_entropy(
        y_pred.view(-1, y_pred.size(-1)),
        y_true.view(-1).long(),
        reduction='none'
    )
    mask = (y_true != 0).float().view(-1)
    loss = (loss * mask).sum() / mask.sum().clamp(min=1)
    return loss * 20  # MLM 학습 강화


def nsp_loss(y_true, y_pred):
    """
    NSP loss
    :param y_true: (bs,)  0 or 1
    :param y_pred: (bs,2) logits
    """
    return F.cross_entropy(y_pred, y_true.long())


def mlm_acc(y_true, y_pred):
    pred_cls = torch.argmax(y_pred, dim=-1).float()
    mask     = (y_true != 0).float()
    correct  = ((y_true.float() == pred_cls) * mask).sum()
    return correct / mask.sum().clamp(min=1)


def nsp_acc(y_true, y_pred):
    pred = torch.argmax(y_pred, dim=-1)
    return (pred == y_true.long()).float().mean()

print("슝=3")

In [ ]:
# ── CosineSchedule ──────────────────────────────────────────

class CosineSchedule:
    """
    Warmup + Cosine Decay LR 스케줄
    """
    def __init__(self, optimizer=None, train_steps=4000, warmup_steps=500, max_lr=2.5e-4):
        self.optimizer    = optimizer
        self.train_steps  = train_steps
        self.warmup_steps = warmup_steps
        self.max_lr       = max_lr
        self.step_num     = 0

    def get_lr(self):
        if self.step_num <= self.warmup_steps:
            return (self.step_num / max(1, self.warmup_steps)) * self.max_lr
        progress = (self.step_num - self.warmup_steps) / \
                   max(1, self.train_steps - self.warmup_steps)
        return 0.5 * self.max_lr * (1 + math.cos(math.pi * progress))

    def step(self):
        self.step_num += 1
        lr = self.get_lr()
        if self.optimizer:
            for g in self.optimizer.param_groups:
                g['lr'] = lr
        return lr


# LR 곡선 시각화
test_sched = CosineSchedule(train_steps=4000, warmup_steps=400)
lrs = [test_sched.step() for _ in range(4000)]

plt.figure(figsize=(8, 3))
plt.plot(lrs, 'r-')
plt.xlabel('Step'); plt.ylabel('LR')
plt.title('Cosine LR Schedule (warmup=400)')
plt.tight_layout(); plt.show()
print("슝=3")

In [ ]:
# ── DataLoader ──────────────────────────────────────────────
BATCH_SIZE = 64

mm = np.memmap(memmap_file, dtype=np.int32, mode='r', shape=(N_INST, 4, N_SEQ))

enc_tokens_all  = torch.from_numpy(mm[:, 0, :].copy())
segments_all    = torch.from_numpy(mm[:, 1, :].copy())
labels_nsp_all  = torch.from_numpy(mm[:, 2, 0].copy())   # shape (N_INST,)
labels_mlm_all  = torch.from_numpy(mm[:, 3, :].copy())

dataset    = TensorDataset(enc_tokens_all, segments_all, labels_nsp_all, labels_mlm_all)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"데이터셋 크기: {len(dataset)}")
print(f"배치 수      : {len(dataloader)}")

In [ ]:
# ── 학습 설정 ────────────────────────────────────────────────
EPOCHS       = 10
WARMUP_RATIO = 0.1

total_steps  = EPOCHS * len(dataloader)
warmup_steps = int(total_steps * WARMUP_RATIO)

model     = MiniBERTPretrain(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=2.5e-4, betas=(0.9, 0.999), eps=1e-8)
scheduler = CosineSchedule(optimizer, train_steps=total_steps, warmup_steps=warmup_steps, max_lr=2.5e-4)

print(f"총 step: {total_steps}, warmup: {warmup_steps}")

In [ ]:
# ── 학습 루프 (10 Epoch) ─────────────────────────────────────
history = {
    'loss': [], 'nsp_loss': [], 'mlm_loss': [],
    'nsp_acc': [], 'mlm_acc': []
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = ep_nsp_loss = ep_mlm_loss = 0.0
    ep_nsp_acc = ep_mlm_acc = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)
    for batch in pbar:
        enc, seg, y_nsp, y_mlm = [b.to(device) for b in batch]

        optimizer.zero_grad()
        logits_nsp, logits_lm = model(enc, seg)

        loss_nsp = nsp_loss(y_nsp, logits_nsp)
        loss_mlm = mlm_loss(y_mlm, logits_lm)
        loss     = loss_nsp + loss_mlm

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()

        ep_loss     += loss.item()
        ep_nsp_loss += loss_nsp.item()
        ep_mlm_loss += loss_mlm.item()
        ep_nsp_acc  += nsp_acc(y_nsp, logits_nsp).item()
        ep_mlm_acc  += mlm_acc(y_mlm, logits_lm).item()

        pbar.set_postfix({
            'loss'    : f"{loss.item():.3f}",
            'nsp_loss': f"{loss_nsp.item():.3f}",
            'mlm_loss': f"{loss_mlm.item():.3f}"
        })

    n = len(dataloader)
    history['loss'].append(ep_loss / n)
    history['nsp_loss'].append(ep_nsp_loss / n)
    history['mlm_loss'].append(ep_mlm_loss / n)
    history['nsp_acc'].append(ep_nsp_acc / n)
    history['mlm_acc'].append(ep_mlm_acc / n)

    print(f"[Epoch {epoch:2d}] loss={history['loss'][-1]:.4f} "
          f"| nsp_loss={history['nsp_loss'][-1]:.4f} mlm_loss={history['mlm_loss'][-1]:.4f} "
          f"| nsp_acc={history['nsp_acc'][-1]:.4f} mlm_acc={history['mlm_acc'][-1]:.4f}")

# 모델 저장
ckpt_path = os.path.join(MODEL_DIR, 'mini_bert_10ep.pt')
torch.save({
    'epoch'      : EPOCHS,
    'model_state': model.state_dict(),
    'optim_state': optimizer.state_dict(),
    'config'     : dict(config),
    'history'    : history
}, ckpt_path)
print(f"\n모델 저장 완료: {ckpt_path}")

---
## Step 7. 프로젝트 결과 시각화

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ① 전체 Loss
axes[0].plot(epochs_range, history['loss'],     'b-o', label='Total Loss')
axes[0].plot(epochs_range, history['nsp_loss'], 'g--s', label='NSP Loss')
axes[0].plot(epochs_range, history['mlm_loss'], 'r--^', label='MLM Loss')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ② NSP Accuracy
axes[1].plot(epochs_range, history['nsp_acc'], 'g-o')
axes[1].set_title('NSP Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='random baseline')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ③ MLM Accuracy
axes[2].plot(epochs_range, history['mlm_acc'], 'r-o')
axes[2].set_title('MLM Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)

plt.suptitle('mini BERT Pretraining Results (vocab=8000, params~1M, 10 Epochs)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'mini_bert_training_curve.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("\n학습 완료!")

In [ ]:
# 최종 결과 요약
print("="*55)
print("          mini BERT 학습 결과 요약")
print("="*55)
print(f"  vocab_size : {config.n_vocab}")
print(f"  d_model    : {config.d_model}")
print(f"  n_layer    : {config.n_layer}")
print(f"  n_head     : {config.n_head}")
print(f"  d_ff       : {config.d_ff}")
print(f"  n_seq      : {config.n_seq}")
total_params = sum(p.numel() for p in model.parameters())
print(f"  총 파라미터 : {total_params:,}")
print("-"*55)
print(f"  최종 Total Loss : {history['loss'][-1]:.4f}")
print(f"  최종 NSP Loss   : {history['nsp_loss'][-1]:.4f}")
print(f"  최종 MLM Loss   : {history['mlm_loss'][-1]:.4f}")
print(f"  최종 NSP Acc    : {history['nsp_acc'][-1]:.4f}")
print(f"  최종 MLM Acc    : {history['mlm_acc'][-1]:.4f}")
print("="*55)